In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 Johnny Nuñez Cano
# SPDX-License-Identifier: MIT
# Relocated from the source revision recorded in ../MIGRATION_MANIFEST.json.
# See ../LICENSE and ../THIRD_PARTY_NOTICES.md.


# Simulating a Robot Arm with MuJoCo

Welcome to the first part of this series. Before we can make simulation *fast*, we need a simulation that is *correct*. In this part you will build a complete robot manipulation task using [MuJoCo](https://mujoco.readthedocs.io/), the CPU physics engine that has underpinned a decade of robot learning research.

The robot is the [SO-101](https://github.com/google-deepmind/mujoco_menagerie/tree/main/robotstudio_so101), a low-cost 5-DoF arm with a parallel gripper from MuJoCo Menagerie. These fundamentals use the original two-cube scene. The next notebook builds the shared pick-and-place task: grasp each cube and place it in a receiving box using physical gripper contacts.

Everything here runs on a single CPU core, in double precision, in one world. That is deliberate: this is the **ground truth** that Part 2 (MJWarp on GPU) and Part 3 (Newton) will be measured against.

## Outline

The progression of this part is as follows:

- An introduction to MuJoCo's three core objects: `MjModel`, `MjData`, and `mj_step`.
- An inspection of the scene we will simulate: the SO-101 arm, a table, and two cubes.
- A hands-on tour of stepping physics, forward kinematics, and inverse kinematics.
- A discussion of the two bugs that make robot grasping fail in practice, and how this scene avoids them.

Part 2 then refactors this exact task to run on thousands of GPU worlds with MJWarp, and Part 3 moves it onto the Newton physics engine.

## What you will learn

By the end of this part, you will be able to:

- Explain the roles of `MjModel`, `MjData`, and `mj_step` in a MuJoCo program.
- Load an MJCF scene, inspect its degrees of freedom, and advance it in time.
- Distinguish forward kinematics from inverse kinematics and use both from Python.
- Identify why MuJoCo on the CPU is latency-optimal but throughput-limited.

## Environment check

Let's confirm the packages for this part are installed. Only `mujoco` and `numpy` are required for Part 1 — no GPU needed yet.

## Open this lesson

Run from the checked-out tutorial in its Python 3.12 environment. In Colab, the setup fetches the selected public source and installs the hash-locked packages only in a fresh Python 3.12 kernel. Hosted execution has not been validated. Existing source files are preserved.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/johnnynunez/accelerated-computing-hub.git"
REF = "cf96302fd7200fac5941aacd9d92b1c2c7b0f386"
TUTORIAL_RELATIVE = Path("tutorials/sim2real-blogs/notebooks/mujoco")
COLAB_CHECKOUT = Path("/content/ach-mujoco-box-source")

if IN_COLAB:
    repo_root = COLAB_CHECKOUT
    git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    if not repo_root.exists():
        repo_root.mkdir(parents=True)
        subprocess.run(["git", "init", str(repo_root)], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "remote", "add", "origin", REPO_URL], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "sparse-checkout", "init", "--cone"], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "sparse-checkout", "set", str(TUTORIAL_RELATIVE)], check=True, env=git_env)
    actual_remote = subprocess.check_output(["git", "-C", str(repo_root), "remote", "get-url", "origin"], text=True).strip()
    if actual_remote != REPO_URL:
        raise RuntimeError("Existing checkout uses another repository; choose a fresh COLAB_CHECKOUT.")
    subprocess.run(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", REF], check=True, env=git_env)
    requested = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "FETCH_HEAD"], text=True).strip()
    existing = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if existing.returncode == 0 and existing.stdout.strip() != requested:
        raise RuntimeError("REF changed; existing files were preserved. Choose a fresh COLAB_CHECKOUT.")
    if existing.returncode != 0:
        subprocess.run(["git", "-C", str(repo_root), "checkout", "--detach", requested], check=True, env=git_env)
else:
    here = Path.cwd().resolve()
    repo_root = next((p for p in (here, *here.parents)
                      if (p / TUTORIAL_RELATIVE / "requirements.lock.txt").is_file()), None)
    if repo_root is None:
        raise RuntimeError("Open the notebook inside the accelerated-computing-hub checkout.")

tutorial_dir = repo_root / TUTORIAL_RELATIVE
lesson_dir = tutorial_dir / "part1"
if not (lesson_dir / "viewer_window.py").is_file():
    raise RuntimeError("This checkout does not contain the current box lessons.")
if IN_COLAB:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("These teaching cells require a Python 3.12 kernel. Use the local/Brev environment; the benchmark notebook provides an isolated Python 3.12 runtime.")
    if any(name in sys.modules for name in ("mujoco", "mujoco_warp", "warp")):
        raise RuntimeError("Restart the runtime before installing the locked physics packages.")
    os.environ.setdefault("MUJOCO_GL", "egl")
    subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes", "-r", str(tutorial_dir / "requirements.lock.txt")], check=True)
os.chdir(lesson_dir)
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
revision = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "HEAD"], capture_output=True, text=True)
print("Source:", revision.stdout.strip() if revision.returncode == 0 else "Image snapshot without Git metadata; retain source fingerprints.")
print("Lesson directory:", lesson_dir)


In [ ]:
import mujoco
import numpy as np

print("mujoco :", mujoco.__version__)
print("numpy  :", np.__version__)

## The scene

Alongside this notebook you will find a few Python files and a `solutions` directory.

| File | What it is |
|------|------------|
| `robots.py` | Profiles for the SO-101 and the Seeed reBot DevArm (`--robot so101|rebot`) |
| `pick_place_common.py` | Shared task logic: scene generation, inverse kinematics, the waypoint controller |
| `utils.py` | Asset download (sparse clone of MuJoCo Menagerie) and macOS viewer helpers |
| `so101_pick_place.py` | **The file you will complete** in the next notebook |
| `solutions/` | The finished script plus one snippet per exercise step |

The pick-and-place MJCF is generated at runtime: it `<include>`s the unmodified robot and only adds the task props. Switch arms with `--robot rebot` without changing the controller.

In [ ]:
from pick_place_common import resolve_pick_place_scene
from robots import get_robot

scene = resolve_pick_place_scene(spec=get_robot("so101"))
print(scene.read_text())

Notice the coordinate convention, which the rest of the series relies on:

- The robot base sits at the world origin and reaches along **+X**.
- The task is laid out across **Y**, so the pick and the place are transverse to the arm's reach.

The robot assets themselves are downloaded on first use (a sparse clone of just that robot's folder, not the whole Menagerie). The next cell resolves them and writes a generated scene next to a copy of the robot XML so all relative paths work.

In [ ]:
from pick_place_common import resolve_pick_place_scene

xml_path = resolve_pick_place_scene()
print("Scene ready:", xml_path)

## MjModel and MjData

Every MuJoCo program is built around three things:

| Object | Mutability | What it holds |
|--------|-----------|---------------|
| `mujoco.MjModel` | read-only | The compiled model: geometry, inertia, joint structure, actuator limits |
| `mujoco.MjData` | mutable | The state: `qpos`, `qvel`, `ctrl`, contacts, simulation time |
| `mujoco.mj_step` | — | Advances `MjData` by one `model.opt.timestep` |

Keeping model and state separate is what makes batching possible later: in Part 2 we upload **one** model to the GPU and allocate **thousands** of states against it.

We load the model through `load_pick_place_model()` rather than calling `mujoco.MjModel.from_xml_path()` directly, so that every part of this series simulates the identical model.

In [ ]:
from pick_place_common import load_pick_place_model

model = load_pick_place_model(xml_path)
data = mujoco.MjData(model)

print(f"nq  (generalized coordinates) : {model.nq}")
print(f"nv  (degrees of freedom)      : {model.nv}")
print(f"nu  (actuators)               : {model.nu}")
print(f"nbody                         : {model.nbody}")
print(f"ngeom                         : {model.ngeom}")
print(f"timestep                      : {model.opt.timestep} s")

`nq` is larger than the number of arm joints because each cube carries a **free joint**: 7 numbers (3 for position, 4 for the orientation quaternion) in `qpos`, but only 6 in `qvel`. That mismatch between `nq` and `nv` is normal in MuJoCo and is the reason positions and velocities are indexed separately.

Let's list the actuators, which are what our controller will drive.

In [ ]:
for i in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    lo, hi = model.actuator_ctrlrange[i]
    print(f"{i}: {name:<14} ctrlrange=[{lo:+.3f}, {hi:+.3f}]")

## Stepping physics

Now let's actually simulate. We put the arm in its grasp-ready home pose, drop the cubes onto the table, and step forward in time.

`apply_arm_ctrl()` writes both `data.ctrl` and the matching `data.qpos` and then runs `mj_forward`, so every body pose is immediately valid. This kind of teleporting is only ever acceptable for **initialization** — once the loop starts, state must evolve through `mj_step`.

In [ ]:
from pick_place_common import HOME_CTRL, apply_arm_ctrl, reset_cubes

apply_arm_ctrl(model, data, HOME_CTRL)
reset_cubes(model, data)

print("t =", data.time)
print("arm qpos :", np.round(data.qpos[:6], 3))

# Let the scene settle under gravity for half a second of simulated time.
n_steps = int(0.5 / model.opt.timestep)
for _ in range(n_steps):
    mujoco.mj_step(model, data)

print(f"\nAfter {n_steps} steps of mj_step:")
print("t =", round(data.time, 4))
print("arm qpos :", np.round(data.qpos[:6], 3))
print("contacts :", data.ncon)

Two things to take away:

1. **`mj_step` is one call per world.** There is no batch dimension anywhere in `MjData`. To simulate 1,000 environments you would need 1,000 `MjData` objects and 1,000 calls — which is exactly the bottleneck Part 2 removes.
2. **`data.ncon` is non-zero**, meaning the solver is resolving real contacts between the cubes and the table. Contact-rich scenes are the expensive case, and also the one MJWarp scales best on.

Let's time a single step to establish the CPU baseline.

In [ ]:
import time

n_timed = 2000
t0 = time.perf_counter()
for _ in range(n_timed):
    mujoco.mj_step(model, data)
elapsed = time.perf_counter() - t0

print(f"{n_timed} steps in {elapsed:.3f} s")
print(f"{elapsed / n_timed * 1e6:.1f} microseconds per step")
print(f"{n_timed / elapsed:,.0f} steps/second (single world)")

Remember that number. MuJoCo on the CPU is **latency-optimal**: a single step is very fast in absolute terms, which is what makes it the right tool for model-predictive control and interactive teleoperation. What it cannot do is produce the millions of steps per second that reinforcement learning needs. That is a throughput problem, and it is what Part 2 is about.

## Forward kinematics and the grasp point

Forward kinematics answers: *given joint angles, where is everything in the world?* MuJoCo computes it inside `mj_step` (and on demand with `mj_forward`), writing results into `data.xpos`, `data.geom_xpos`, and `data.site_xpos`.

Here is the first bug that costs people real debugging time. The SO-101 model ships with a `gripperframe` site, and it is tempting to treat it as the grasp point. It is not: it sits roughly 17 mm away from the actual gap between the fingers. Drive inverse kinematics to the site and your gripper will consistently *miss* the object.

The fix is to measure the true aperture center as the midpoint between the fixed and moving jaw tips.

In [ ]:
from pick_place_common import aperture_center

mujoco.mj_forward(model, data)

site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "gripperframe")
site_pos = data.site_xpos[site_id]
grasp_pos = aperture_center(model, data)

print("gripperframe site :", np.round(site_pos, 4))
print("aperture center   :", np.round(grasp_pos, 4))
print("offset            :", round(float(np.linalg.norm(site_pos - grasp_pos)) * 1000, 1), "mm")

## Inverse kinematics

Inverse kinematics is the reverse question: *given a desired world position, what joint angles get me there?* There is no closed-form answer for a general arm, so we solve it iteratively with **damped least squares**:

```
repeat:
    run FK at the current guess
    err = target - aperture_center
    J   = positional Jacobian of the aperture point   (mujoco.mj_jac)
    dq  = J^T (J J^T + lambda I)^-1 err
    q  += step * dq
    clamp q to joint limits
```

Two design choices in `pick_place_common.ik_to_target()` are worth understanding:

- **The wrist roll is locked.** The SO-101 has 5 arm degrees of freedom, which cannot reach an arbitrary 6-DoF pose. We fix the redundant joint at the Menagerie *pickup* keyframe value so the jaws open horizontally and can pinch the cube's side faces, then solve position-only IK for the remaining four joints.
- **Damping (lambda) keeps the solve stable** near singularities, at the cost of a small steady-state error. That trade is the right one here.

Let's ask the solver to move the grasp point 8 cm above the red cube.

In [ ]:
from pick_place_common import GRIPPER_OPEN, RED_CUBE_POS, ik_to_target

target = RED_CUBE_POS + np.array([0.0, 0.0, 0.08])

ik_data = mujoco.MjData(model)
apply_arm_ctrl(model, ik_data, HOME_CTRL)

ctrl = ik_to_target(model, ik_data, target, HOME_CTRL.copy(), gripper=GRIPPER_OPEN)
apply_arm_ctrl(model, ik_data, ctrl)
reached = aperture_center(model, ik_data)

print("target   :", np.round(target, 4))
print("reached  :", np.round(reached, 4))
print("residual :", round(float(np.linalg.norm(target - reached)) * 1000, 2), "mm")
print("ctrl     :", np.round(ctrl, 3))

A sub-millimetre residual means the solver converged. Note that IK is pure **kinematics** — it never touched the physics state. That property is why Parts 2 and 3 reuse this controller unchanged while the dynamics backend changes underneath.

## The second bug: grabbing a free object

The other lesson that costs debugging time is the grasp itself. If you command the gripper from open to closed in a single step, the moving jaw accelerates into the cube and shoves it away before friction can take hold. The cube skitters off the table and the grasp fails.

The fix is to **ramp the close gradually**. In `pick_place_common.build_auto_sequence()` the `grasp_red` waypoint has a deliberately long duration so the jaws squeeze rather than slam.

In [ ]:
from pick_place_common import build_auto_sequence

for wp in build_auto_sequence():
    print(f"{wp.name:<12} target={np.round(wp.target, 3)}  gripper={wp.gripper:.2f}  {wp.duration:.1f}s")

## Render a frame

Finally, let's look at the scene. MuJoCo's offscreen `Renderer` needs a working OpenGL context; if this cell fails in your environment, skip it and use the interactive viewer from the terminal instead (covered in the next notebook).

In [ ]:
try:
    import matplotlib.pyplot as plt

    apply_arm_ctrl(model, data, HOME_CTRL)
    reset_cubes(model, data)

    with mujoco.Renderer(model, height=480, width=640) as renderer:
        camera = mujoco.MjvCamera()
        mujoco.mjv_defaultCamera(camera)
        camera.azimuth, camera.elevation, camera.distance = 140, -25, 1.0
        camera.lookat[:] = [0.35, -0.04, 0.12]
        renderer.update_scene(data, camera)
        frame = renderer.render()

    plt.figure(figsize=(8, 6))
    plt.imshow(frame)
    plt.axis("off")
    plt.title("SO-101 at the home pose")
    plt.show()
except Exception as exc:  # noqa: BLE001 - rendering is environment dependent
    print("Offscreen rendering unavailable in this environment:")
    print(f"  {type(exc).__name__}: {exc}")
    print("\nUse the interactive viewer instead (from a terminal):")
    print("  python so101_pick_place.py  # add --robot rebot for the Seeed DevArm")

## Recap

Before moving on, make sure you can answer:

- What is the difference between `MjModel` and `MjData`, and why does the split matter for GPU batching?
- Why is `nq` larger than `nv` in this scene?
- Why do we drive inverse kinematics to the aperture center instead of the `gripperframe` site?
- Why does the grasp waypoint use a long duration?

## Next

Continue to **`02__pick_and_place.ipynb`**, where you will complete `so101_pick_place.py` and run the full task end to end.